# 03 - RAG retrieval evaluation

MRR@3, Hit@1 and negative-case accuracy of the module retriever against `data/ground_truth/ground_truth.json`. The default embedder here is the offline hashing one so the notebook runs anywhere; switch `EMBEDDER` to compare with MiniLM or Titan.

> The ground-truth file is **seed data** for a five-module corpus, so MRR saturates. Replace it with expert-labelled real findings before reporting.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))   # make the project packages importable from the notebook

In [ ]:
import tempfile
import pandas as pd

from evaluation.mrr_evaluator import evaluate_mrr, load_ground_truth
from rag_pipeline.module_indexer import get_embedder, index_modules
from rag_pipeline.retriever import ModuleRetriever

EMBEDDER = "hash"          # "hash" | "default" (Chroma MiniLM) | "bedrock" (Titan)
chroma = tempfile.mkdtemp(prefix="chroma_eval_")
embedder = get_embedder(EMBEDDER)
print("modules indexed:", index_modules(ROOT / "terraform_modules", chroma, embedder))
retriever = ModuleRetriever(chroma_path=chroma, embedder=embedder)
entries = load_ground_truth(ROOT / "data" / "ground_truth" / "ground_truth.json")
len(entries)

In [ ]:
result = evaluate_mrr(retriever, entries, k=3)
pd.Series({k: v for k, v in result.items() if k != "per_item"})

## Where does retrieval go wrong?

In [ ]:
per_item = pd.DataFrame(result["per_item"])
per_item["relevant"] = per_item["relevant"].apply(", ".join)
per_item["ranked"] = per_item["ranked"].apply(", ".join)
wrong = per_item[(per_item["rr"] < 1.0) | ((per_item["rr"].isna()) & (per_item["ranked"] != ""))]
wrong if len(wrong) else "no retrieval errors"

## Reciprocal rank by module

In [ ]:
gt = pd.DataFrame(entries)[["id", "control_id", "title", "resource_type", "relevant_modules"]]
gt["module"] = gt["relevant_modules"].apply(lambda m: m[0] if m else "(none)")
merged = gt.merge(per_item[["id", "rr"]], on="id")
merged.groupby("module")["rr"].agg(["count", "mean"]).round(3)

## Sensitivity to `min_similarity`
Higher thresholds reject more unrelated findings (better negative accuracy) but can drop true matches that lack a resource-type or control-ID match.

In [ ]:
rows = []
for threshold in (0.1, 0.2, 0.3, 0.4, 0.5):
    r = ModuleRetriever(chroma_path=chroma, embedder=embedder, min_similarity=threshold)
    res = evaluate_mrr(r, entries, k=3)
    rows.append({"min_similarity": threshold, "mrr": res["mrr"], "hit_at_1": res["hit_at_1"], "no_module_accuracy": res["no_module_accuracy"]})
pd.DataFrame(rows).round(3)